In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableBranch,RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from typing import List,Literal
from pydantic import BaseModel,Field


In [71]:
class Sentiment(BaseModel):
    sentiment: Literal['Positive','Negative']=Field(description='Generated  sentiment MUST be Positive or Negative only')

In [72]:
parser=PydanticOutputParser(pydantic_object=Sentiment)

In [73]:
prompt_template=PromptTemplate(template='Generate the Sentiment of the following {feedback} in {format}',input_variables=['feedback'],partial_variables={'format':parser.get_format_instructions()})

In [74]:
openai=ChatOpenAI(model='gpt-4o-mini')

In [75]:
chain1=prompt_template | openai | parser

In [ ]:
# res1=chain1.invoke({'feedback':'I felt very bad yesterday'})

In [ ]:
# res1.sentiment

'Negative'

In [76]:
prompt_template1=PromptTemplate(template='Generate a positive response for the received {Sentiment} to the user',input_variables='Sentiment')

In [78]:
prompt_template2=prompt_template1=PromptTemplate(template='Generate a response as sorry message for the received {Sentiment} to the user',input_variables='Sentiment')

In [79]:
gemini=ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [80]:
str_parser=StrOutputParser()

In [48]:
conditional_chain=RunnableBranch((lambda x:x.sentiment=='Positive',prompt_template1|gemini|str_parser),
                                   (lambda x:x.sentiment =='Negative',prompt_template1|gemini|str_parser) ,
                                   RunnableLambda(lambda x:'Could not able to extract the Sentiment from feedback'))


In [81]:
conditional_chain_openai=RunnableBranch((lambda x:x.sentiment=='Positive',prompt_template1|openai|str_parser),
                                   (lambda x:x.sentiment =='Negative',prompt_template1|openai|str_parser) ,
                                   RunnableLambda(lambda x:'Could not able to extract the Sentiment from feedback'))


In [82]:
final_chain=chain1| conditional_chain_openai

In [85]:
result1=final_chain.invoke({'I am at home today'})

In [86]:
result1

"Subject: Our Sincere Apologies\n\nDear [User's Name],\n\nThank you for your positive feedback! We’re thrilled to hear that you had a great experience with us. Your kind words motivate us to keep improving and providing the best service possible.\n\nHowever, we sincerely apologize if there was any aspect of your experience that did not meet your expectations. We value your input as it helps us to enhance our services.\n\nIf you have any further thoughts or suggestions, please feel free to share them with us. We truly appreciate your support and look forward to serving you even better in the future!\n\nBest regards,\n\n[Your Name]  \n[Your Position]  \n[Your Company]  "

In [88]:
final_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
